# Autonomous Code Debugging Agent

This notebook uses the modular package in `src/autonomous_code_debugging_agent` to analyze repositories and retrieve likely fix targets.

1. Run the setup cell.
2. Point `repo_path` at a local repository or GitHub URL.
3. Run the analysis cell to retrieve relevant code and ask the LLM for a debugging suggestion.


In [ ]:
import pathlib
import sys

# Make the local package importable when the notebook is started from the repo root.
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))

from autonomous_code_debugging_agent import analyze_repository

print('Notebook ready')


Notebook ready


In [ ]:
repo_path = pathlib.Path.cwd()
error_message = 'division by zero'

# Use the current project folder as the default repository to inspect.
print(f'Analyzing repository: {repo_path}')


Analyzing repository: c:\Users\Siddharth\Downloads\Autonomous_Code_Debugging_Agent


In [ ]:
report = analyze_repository(str(repo_path), error_message, top_k=3)

# analyze_repository returns the matched chunks plus the LLM suggestion.
print('Retrieved chunks:', len(report['results']))
for index, result in enumerate(report['results'], start=1):
    # Each result contains a similarity score and the actual code snippet text.
    print(f'--- {index} score={result["score"]:.4f} ---')
    print(result['text'])

print('LLM suggestion:')
print(report['suggestion'])


Retrieved chunks: 3
--- 1 score=0.1073 ---
    def test_analyze_repository_with_html_report_writes_file(self):
        with tempfile.TemporaryDirectory() as temp_dir:
            repo_root = Path(temp_dir)
            (repo_root / "sample.py").write_text(
                "def divide(a, b):\n    if b == 0:\n        raise ZeroDivisionError\n    return a / b\n",
                encoding="utf-8",
            )

            report = analyze_repository_with_html_report(
                str(repo_root),
                "ZeroDivisionError: division by zero",
                top_k=1,
                output_dir=Path(temp_dir),
            )

            report_path = Path(report["report_path"])
            self.assertTrue(report_path.exists())
            html = report_path.read_text(encoding="utf-8")
            self.assertIn("ZeroDivisionError: division by zero", html)
            self.assertIn("sample.py", html)
--- 2 score=0.0916 ---
    def test_retriever_returns_most_relevant_chunk(self):
 

In [ ]:
def run_debug_loop(repo_path: str, error_message: str, max_iters: int = 2):
    current_error = error_message
    for iteration in range(1, max_iters + 1):
        # Re-run analysis on the latest error text so the loop can refine the suggestion.
        print(f'=== Iteration {iteration}/{max_iters} ===')
        report = analyze_repository(repo_path, current_error, top_k=3)

        # Show the most similar snippet first because it usually contains the likely failure site.
        print('Top relevant snippet:')
        print(report['results'][0]['text'][:600] if report['results'] else 'No relevant snippets found.')

        # Feed the previous model suggestion back as the next query string.
        print('\nSuggestion:')
        print(report['suggestion'])
        current_error = report['suggestion']

run_debug_loop(str(repo_path), error_message, max_iters=2)


=== Iteration 1/2 ===
Top relevant snippet:
    def test_html_report_generator_writes_summary_file(self):
        with tempfile.TemporaryDirectory() as temp_dir:
            report = {
                "snapshot": {"files": ["src/example.py"]},
                "results": [
                    {
                        "text": "def divide(a, b):\n    if b == 0:\n        raise ZeroDivisionError\n    return a / b",
                        "metadata": {
                            "path": "src/example.py",
                            "name": "divide",
                            "kind": "function",
                        },
                

Suggestion:
No LLM suggestion available; set OPENAI_API_KEY or GEMINI_API_KEY.
=== Iteration 2/2 ===
Top relevant snippet:
# Autonomous Code Debugging Agent

A desktop-friendly repository analysis tool that inspects a project, validates dependencies, and produces debugging insights with a polished dashboard.

## What this project does

- Loads a reposi